Retrieve libraries and link Google Drive with Colab

In [ ]:
import tensorflow as tf
from keras import layers
import matplotlib.pyplot as plt
from google.colab import drive
import os

# Linking the drive
drive.mount('/content/drive')

# Create a dedicated folder for the project in Drive to ensure organization
save_dir = "/content/drive/MyDrive/Dog_Muffin_Project"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

checkpoint_path = os.path.join(save_dir, "best_vit_model.keras")
print(f"The model will be saved in: {checkpoint_path}")

Download training data from the Drive path

In [ ]:
# Track and size settings
data_path = '/content/drive/MyDrive/Muffinvs.Chihuahua/train'
IMG_SIZE = 224
BATCH_SIZE = 16
# Download training data (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)
# Upload verification data (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"The types of items found: {class_names}")

Creating the model

In [ ]:
def create_vit_classifier_v2():
    patch_size = 8
    projection_dim = 128
    num_heads = 8
    transformer_layers = 12
    num_patches = (IMG_SIZE // patch_size) ** 2

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),])
    # 1. Standardization
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # 2. Embedding
    x = layers.Conv2D(filters=projection_dim, kernel_size=patch_size, strides=patch_size, padding="valid")(x)
    x = layers.Reshape((num_patches, projection_dim))(x)

    positions = tf.range(start=0, limit=num_patches, delta=1)
    position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)(positions)
    x = x + position_embedding
    #3. Transformer Layers
    for _ in range(transformer_layers):
        # Dropout to prevent overfitting
        y = layers.LayerNormalization(epsilon=1e-6)(x)
        y = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=0.1)(y, y)
        x = layers.Add()([y, x])

        # Feed Forward
        y = layers.LayerNormalization(epsilon=1e-6)(x)
        y = layers.Dense(projection_dim * 2, activation="gelu")(y)
        y = layers.Dropout(0.1)(y)
        y = layers.Dense(projection_dim, activation="gelu")(y)
        x = layers.Add()([y, x])
    # 4. Final Ranking
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(len(class_names))(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

model = create_vit_classifier_v2()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseCategoricalAccuracy(name="f1_proxy")
    ]
)

Start training the model and save the model with each attempt to protect against loss.

In [ ]:
# 1. Setting up callbacks for protection
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
)
# 2. Resume Logic
if os.path.exists(checkpoint_path):
    print("A previous model has been found, uploading to complete...")
    model = tf.keras.models.load_model(checkpoint_path)
else:
    print("There is no previous model; we will start training from scratch.")

# 3. Start training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    batch_size=16,
    callbacks=[checkpoint_callback, lr_reducer]
)

Keeping final weights is also important as a precaution.

In [ ]:
model.save_weights('/content/drive/MyDrive/vit_dog_vs_muffin.weights.h5')
print("The final weights have been successfully saved to Google Drive!")


Reloading weights

In [ ]:
model_path = '/content/drive/MyDrive/vit_dog_vs_muffin.weights.h5'

model.load_weights(model_path)

print("The model was successfully restored")

Download test data and start the test

In [ ]:
import os
import pandas as pd
import numpy as np
from keras.preprocessing import image

test_folder_path = '/content/drive/MyDrive/Muffinvs.Chihuahua/kaggle_test_final'
output_csv_path = '/content/drive/MyDrive/submission.csv'

class_names = ['chihuahua', 'muffin']

results = []

print("Images are being processed and predictions are being extracted....")

# Name order to ensure formatting
image_files = sorted([f for f in os.listdir(test_folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))])

for img_name in image_files:
    img_path = os.path.join(test_folder_path, img_name)

    # Image preparation
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # Expectation
    prediction = model.predict(img_array, verbose=0)
    if prediction.shape[1]>0:
      idx=np.argmax(prediction[0])
    else:
      idx = int(round(prediction[0][0]))
    idx=min(idx,len(class_names)-1)
    label = class_names[idx]

    # Add the result to the list
    results.append({'ID': img_name, 'Label': label})

# Convert the results to a DataFrame and save them in CSV format
df = pd.DataFrame(results)
df.to_csv(output_csv_path, index=False)


# Show the first 5 lines to confirm
print("\n Sample results:")
print(df.head())